# ARIMA - Tourist Arrivals Forecast

## Import Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_percentage_error


## Load Dataset

In [ ]:
df = pd.read_csv("tourism_data_2005_onwards (1).csv")
df.head()


## Shape

In [ ]:
df.shape


## Convert Month to Time-Series Index

In [ ]:
df["Month"] = pd.to_datetime(df["Month"], dayfirst=True)
df = df.sort_values("Month")
df["Month"] = df["Month"].dt.to_period("M").dt.to_timestamp()
df = df.set_index("Month")

tourists = df["Tourists"].astype(float)
tourists.head()


## Plot Monthly Tourist Arrivals

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(tourists)
plt.title("Monthly Tourist Arrivals")
plt.xlabel("Month")
plt.ylabel("Tourists")
plt.show()


## Time-Series Decomposition

In [ ]:
result = seasonal_decompose(tourists, model="additive", period=12)
result.plot()
plt.show()


## Train-Test Split

In [ ]:
train = tourists[:-24]
test = tourists[-24:]

print("Training data:", len(train))
print("Testing data :", len(test))


## Build ARIMA Model

In [ ]:
model = ARIMA(train, order=(1, 1, 1))
model_fit = model.fit()

print(model_fit.summary())


## Forecast

In [ ]:
forecast = model_fit.forecast(steps=len(test))
forecast.index = test.index

forecast


## Calculate MAPE

In [ ]:
mape = mean_absolute_percentage_error(test, forecast) * 100
print("MAPE:", round(mape, 2), "%")


## Actual vs Forecast

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(test.index, test, label="Actual")
plt.plot(forecast.index, forecast, label="Forecast")
plt.title("Actual vs ARIMA Forecast")
plt.xlabel("Month")
plt.ylabel("Tourists")
plt.legend()
plt.show()


## Interpretation

In [ ]:
if mape < 10:
    print("The forecast has good accuracy.")
else:
    print("The forecast has higher error.")


## Actual and Forecast Table

In [ ]:
result_df = pd.DataFrame({"Actual": test, "Forecast": forecast})
result_df.round(2)
